In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

#Preprocessing
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler

#Classification
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier

#Tuning and Evaluation
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Create spark session
spark = SparkSession.builder \
    .appName("IrisClassification") \
    .getOrCreate()

In [2]:
#Read Iris dataset with header
df = spark.read.csv("iris.csv", header=True, inferSchema=True)

In [3]:
# Data Description summary
print(df.printSchema())
print(df.show())
df.describe().show()

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)

None
+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
|         5.4|        3.9|         1.7|        0.4| setosa|
|         4.6|        3.4|         1.4|        0.3| setosa|
|         5.0|        3.4|         1.5|        0.2| setosa|
|         4.4|        2.9|         1.4|        0.2| setosa|
|         4.9|        3.1|         1.5|        0.1| seto

### Interpretation of Data 
+ Through the schema of the features, it is clearly to see that **sepal_length, sepal_width, petal_length and petal_width** are continuous numerical variables. Additionally, **species** is the categorical variable.
+ From the summary of dataset, there is no missing values which indicated that no data cleaniing is required before conducting the analysis.
+ Sepal length has a mean of approximately 5.84 cm with moderate variation, while sepal width is more consistently distributed around a mean of 3.06 cm with lower variability.
+ Petal measurements have a greater variation than sepal width, especially petal length, which has a much higher standard deviation. This suggests that petal features perform better at differentiating across iris species and are likely to be significant for classification models like logistic regression and decision trees.


In [4]:
# Data Preprocessing
label_indexer = StringIndexer(inputCol="species",outputCol="label")

In [47]:
# Combine numeric measurement columns to single vector column
assembler = VectorAssembler(
    inputCols=["sepal_length","sepal_width","petal_length","petal_width"],
    outputCol="features")

In [6]:
# Train-test split in ratio 8:2
train_df, test_df = df.randomSplit([0.8, 0.2], seed=123)

## Logistic Regression Model

In [7]:
# Logistic Regression Model
lr = LogisticRegression(featuresCol="features",labelCol="label")

# Build Pipeline
pipeline_lr = Pipeline(stages=[label_indexer,assembler,lr])

In [8]:
# Hyperparameter Tuning
paramGrid_lr = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

In [39]:
# Cross Validation
cv_lr = CrossValidator(
    estimator=pipeline_lr,
    estimatorParamMaps=paramGrid_lr,
    evaluator=evaluator,
    numFolds=3)

+ Combines into 9 total hyperparameter grids and 3 folds, total 27 runs of model evaluated. This is to ensure the final chosen models are empirically optimized against validation splits.

In [11]:
# Train Model
cvModel_lr = cv_lr.fit(train_df)

In [35]:
# Prediction on test set
predictions_lr = cvModel_lr.transform(test_df)

In [16]:
# Evaluation of Logistic Regression
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy")

precision_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision")

recall_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall")

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1")

accuracy_lr = accuracy_eval.evaluate(predictions_lr)
precision_lr = precision_eval.evaluate(predictions_lr)
recall_lr = recall_eval.evaluate(predictions_lr)
f1_lr = f1_eval.evaluate(predictions_lr)

print("Accuracy:", accuracy_lr)
print("Precision:", precision_lr)
print("Recall:", recall_lr)
print("F1-Score:", f1_lr)

Accuracy: 0.9655172413793104
Precision: 0.9689655172413794
Recall: 0.9655172413793104
F1-Score: 0.9650984224486947


## Decision Tree Classification

In [14]:
# Decision Tree Model
dt = DecisionTreeClassifier(featuresCol="features",labelCol="label")

# Build Pipeline
pipeline_dt = Pipeline(stages=[label_indexer,assembler,dt])

In [15]:
# Hyperparameter Grid
paramGrid_dt = ParamGridBuilder() \
    .addGrid(dt.maxDepth, [2, 5, 10]) \
    .addGrid(dt.maxBins, [16, 32, 64]) \
    .build()

In [40]:
# Cross Validation
cv_dt = CrossValidator(
    estimator=pipeline_dt,
    estimatorParamMaps=paramGrid_dt,
    evaluator=evaluator,
    numFolds=3)

+ Combines into 9 total hyperparameter grids and 3 folds, total 27 runs of model evaluated. This is to ensure the final chosen models are empirically optimized against validation splits.

In [18]:
# Train Model
cvModel_dt = cv_dt.fit(train_df)

In [51]:
# Predictions on test model
predictions_dt = cvModel_dt.transform(test_df)

In [27]:
# Evaluation of Decision Tree Model
accuracy_dt = accuracy_eval.evaluate(predictions_dt)
precision_dt = precision_eval.evaluate(predictions_dt)
recall_dt = recall_eval.evaluate(predictions_dt)
f1_dt = f1_eval.evaluate(predictions_dt)

print("Accuracy:", accuracy_dt)
print("Precision:", precision_dt)
print("Recall:", recall_dt)
print("F1-Score:", f1_dt)

Accuracy: 0.9310344827586207
Precision: 0.9310344827586207
Recall: 0.9310344827586207
F1-Score: 0.9310344827586207


## Random Forest Classification

In [22]:
# Random Forest Model
rf = RandomForestClassifier(featuresCol="features",labelCol="label")

# Build Pipeline
pipeline_rf = Pipeline(stages=[label_indexer,assembler,rf])

In [37]:
# Hyperparameter Grid
paramGrid_rf = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20]) \
    .addGrid(rf.maxDepth, [2, 5]) \
    .build()

In [38]:
# Cross Validation
cv_rf = CrossValidator(
    estimator=pipeline_rf,
    estimatorParamMaps=paramGrid_rf,
    evaluator=evaluator,
    numFolds=3)

+ Combines into 4 total hyperparameter grids and 3 folds, total 12 runs of model evaluated. This is to ensure the final chosen models are empirically optimized against validation splits.

In [25]:
# Train Model
cvModel_rf = cv_rf.fit(train_df)

In [26]:
# Prediction on Test Model
predictions_rf = cvModel_rf.transform(test_df)

In [28]:
# Evaluation
accuracy_rf = accuracy_eval.evaluate(predictions_rf)
precision_rf = precision_eval.evaluate(predictions_rf)
recall_rf = recall_eval.evaluate(predictions_rf)
f1_rf = f1_eval.evaluate(predictions_rf)

print("Accuracy:", accuracy_rf)
print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1-Score:", f1_rf)

Accuracy: 0.9655172413793104
Precision: 0.9689655172413794
Recall: 0.9655172413793104
F1-Score: 0.9650984224486947


In [43]:
# Comparison of Model Performance
results = [
    ("Logistic Regression", accuracy_lr, precision_lr, recall_lr, f1_lr),
    ("Decision Tree", accuracy_dt, precision_dt, recall_dt, f1_dt),
    ("Random Forest", accuracy_rf, precision_rf, recall_rf, f1_rf)]

print(results)

[('Logistic Regression', 0.9655172413793104, 0.9689655172413794, 0.9655172413793104, 0.9650984224486947), ('Decision Tree', 0.9310344827586207, 0.9310344827586207, 0.9310344827586207, 0.9310344827586207), ('Random Forest', 0.9655172413793104, 0.9689655172413794, 0.9655172413793104, 0.9650984224486947)]


## Comparative Analysis across Three Models:

### Performance Comparison of each Evaluation Metrics
The three machine learning models were evaluated using four common classification metrics: Accuracy, Precision, Recall, and F1-Score. The results are shown below.

| Model | Accuracy | Precision | Recall | F1-score |
|---|---|---|---|---|
| Logistic Regression | 96.55% | 96.89% | 96.55% | 96.51% |
| Decision Tree | 93.10% | 93.10% | 93.10% | 93.10% |
| Random Forest | 96.55% | 96.89% | 96.55% | 96.51% |

From the results, both Logistic Regression and Random Forest achieved the highest performance, with an accuracy of 96.55% and an F1-score of 96.51%. In comparison, the Decision Tree model conducted lower performance, which are 93.10% across all evaluation metrics.

The findings indicate that Iris dataset can be classified effectively using both linear and ensemble-based classifier approaches. However, the Decision Tree was slightly less accurate.


### Strengths and Limitations of three Models
#### 1. Logistic Regression
Strengths
+ Logistic Regression is a simple and efficient model that performs excellently when the data is approximately linearly separated. Additionally, it is fairly interpretable because model coefficients can be used to interpret the relationship between features and predictions.

Limitations
+ In contrast, the main limitations of logistic regression are it may struggle with complex non-linear patterns in data as it assumes aa linear relationship betwenn features and target classes. Also, the performance may decrease when the dataset contains highly complicated decision boundaries.

#### 2. Decision Tree Classification
Strengths
+ Decision Tree model is easy to understand and visualize, and it can naturally handle non-linear relationships without requiring feature scaling.

Limitations
+ This model is prone to overfitting and can produce unstable results when small changes occur in the dataset.

#### 3. Random Forest Classification
Strength
+ Random Forest model can reduce overfitting by combning multiple decision trees. Also, it provides high accuracy and stable predictions. Furthermore, this model handles noisy and complex datasets effectively.

Limitations
+ On the other hand, random forest model is more computationally expensive than Logistic Regression and Decision Trees. Also, it is hard to interpret as predictions come from many trees. As a result, it requires more memory and processing power.

### Justification of the Best-Performing Model
With an accuracy of 96.55% and an F1-score of 96.51%, Random Forest and Logistic Regression both performed effectively. In contrast, the Decision Tree model performed less effectively, which scored 93.10% on all evaluation metrics.

Although Logistic Regression and Random Forest conducted the same evaluation scores, Logistic Regression is considered as the most suitable model for this dataset. The main reason is that the Iris dataset is relatively simple and mostly linearly separable. Therefore, a complex ensemble method such as Random Forest does not provide a significant performance advantage over Logistic Regression.

Compared to more complex models, it requires less processing power and trains quicker, making the model extremely efficient. It also provides better understanding since its coefficients clearly demonstrate how each petal and sepal measurement influences the final classification outcome.